# Tarea Práctica: Riesgos de Integridad en Entornos Distribuidos

## Contexto
Trabajas como Analista de Datos en un *E-commerce* de rápido crecimiento. El sistema de procesamiento de datos es distribuido: eventos de navegación y transacciones llegan continuamente desde microservicios separados (inventario, carrito, pasarela de pagos) a través de un sistema de mensajería (como Kafka).

Al ser una arquitectura distribuida (con retardos de red, reintentos automáticos y equipos de desarrollo separados), empiezan a aparecer comportamientos anómalos en los registros analíticos diarios.

## Objetivo
Tu objetivo no es la simple "limpieza por valores atípicos" (como hicimos en IoT), sino reparar **riesgos críticos de integridad técnica y lógica** en un *subset* representativo de los datos.

Debes abordar los siguientes problemas extraídos de la **UT3 - Capítulo 2**:
1.  **Duplicados y Replays:** El broker de mensajería reintenta envíos si falla la red, inyectando transacciones repetidas.
2.  **Estados Parciales:** Fallos en el pipeline de enriquecimiento hacen que algunas compras lleguen sin asociar al usuario (`user_id`).
3.  **Corrupción Lógica Silenciosa:** Un cambio silencioso de esquema en origen provoca que números lleguen como textos con formato europeo (ej: `"14,50 €"`), rompiendo los cálculos y devolviendo `NaN` tras conversiones fallidas.

### ¡IMPORTANTE! ⚠️
Como siempre, **justifica** por qué tomas cada decisión (¿borras un nulo? ¿rellenas? ¿te fías de qué columna para borrar duplicados?). Si delegas la justificación exclusivamente a una IA y no refleja tu entendimiento de la teoría, se notará y mas cuando te pregunte en clase.🥸 No me seas, y se curioso y pierde un poco el tiempo en amprender. 


In [ ]:
import pandas as pd
import numpy as np

# --- CÓDIGO DE GENERACIÓN DE DATOS (NO MODIFICAR) ---
# Tienes una muestra de 8 eventos de la última hora en la Capa Bronce (Raw).

data = {
    'event_id': ['EV-100', 'EV-101', 'EV-101', 'EV-102', 'EV-103', 'EV-104', 'EV-105', 'EV-106'],
    'timestamp': [
        '2026-03-01 14:00:00', 
        '2026-03-01 14:05:00', 
        '2026-03-01 14:05:00',  # Reintento exacto del EV-101
        '2026-03-01 14:10:00',
        '2026-03-01 14:12:00',
        '2026-03-01 14:15:00',
        '2026-03-01 14:10:00',  # Evento rezagado (old timestamp)
        '2026-03-01 14:20:00'
    ],
    'user_id': ['U-01', 'U-02', 'U-02', None, 'U-03', 'U-01', 'U-04', 'U-05'], # El EV-102 perdió la identidad del usuario
    'action': ['view', 'purchase', 'purchase', 'purchase', 'view', 'purchase', 'view', 'purchase'],
    'amount': [0.0, 50.5, 50.5, 120.0, 0.0, "25,99 €", 0.0, -10.0] # Falla esquema y valor negativo
}

df_events = pd.DataFrame(data)
print("--- Dataset Raw (Sucio) extraído de Data Lake ---")
display(df_events)

---
### Ejercicio 1: Idempotencia y Duplicados por Reintentos (Replays)
*(Ref: UT3 punto 2.1c)*

**Problema:** En sistemas distribuidos, si el consumidor no confirma haber recibido la transacción `EV-101`, el productor la reenvía *"por si acaso"*. Si sumamos directamente los ingresos, estaríamos contando más dinero del real.

**Tarea:** 
1. Detecta qué fila está duplicada, garantizando la idempotencia basándote exclusivamente en el identificador único de la transacción (`event_id`).
2. Elimina la fila fantasma manteniendo de forma segura la original.

In [ ]:
# TU CÓDIGO AQUÍ



---
### Ejercicio 2: Estados Parciales y Registros Huérfanos
*(Ref: UT3 punto 2.1e)*

**Problema:** El evento `EV-102` es una compra real (`amount = 120.0`) pero se ha quedado en **estado parcial**. Existe, pero le falta el `user_id` asociado porque un microservicio que enriquece los datos de cliente estaba caído.

**Tarea:**
1. A diferencia del caso del "Edificio Inteligente" donde la falta de temperatura se podía interpolar, imputar un `user_id` a la ligera inventándolo puede arruinar métricas críticas como el Ticket Medio por Usuario o cruces de facturación.
2. Demuestra cómo identificarías las compras "huérfanas" (`action == 'purchase'` sin `user_id`).
3. Elimina o mueve ese registro a un dataframe de "cuarentena" justificando tu elección.

In [ ]:
# TU CÓDIGO AQUÍ



---
### Ejercicio 3: Cambio de Esquema y Corrupción Silenciosa Lógica
*(Ref: UT3 punto 2.2c y 2.1b)*

**Problema:** Uno de los equipos front-end hizo una actualización silenciosa y empezó a enviar los importes concatenados con texto español (`"25,99 €"`) en la compra `EV-104`.
Al mezclarse números y Strings `amount` pasa a ser una columna `Object`. También hay un valor sin sentido (`-10.0`).

**Tarea:**
1. Muestra el tipo de dato (`dtypes`) actual de la columna `amount`.
2. Repara la columna: limpia el texto (cambia la coma por punto y quita el " €") solo en las celdas que sean texto, o fuerza a que toda la columna se convierta a `float`. `pd.to_numeric(errors='coerce')` puede ser tu gran aliado táctico.
3. Las compras no pueden tener costes negativos. Localiza el valor numérico contaminado y arréglalo pasándolo a 0 o borrando la fila.

In [ ]:
# TU CÓDIGO AQUÍ



---
### Resultado Final Consolidado
Prueba a sumar todos los `amount` de la columna de las compras resultantes. Tu contabilidad debería ser perfecta ahora.

In [ ]:
# TU CÓDIGO AQUÍ

